# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedfurqan1/FlyRank-MachineLearning/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

###Answer
**Ranking and scoring.**  
For Lane 2, the goal is not simply to classify whether a page is declining in isolation, but to rank declining pages by priority so an editorial team knows which ones to review first. Because human review capacity is limited, generating a continuous priority score allows us to sort the catalog and surface the highest-impact opportunities at the top of the queue.


---



## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

###Answer
I am predicting whether a page is in active decline (labeled as 1 for declining, 0 for healthy).  
In our local starter code, this label comes from a defined rule proxy (trend_direction == "down").   
When we move to the daily warehouse data, this will be replaced with a real observed outcome (whether organic impressions drop by over 20% in the next 30 days).  
To prevent target leakage, metrics derived from the target trend like trend_direction and trend_pct are strictly excluded from the model's input features.

---



## 3. Success metric

*One metric you can defend. What number means 'good'?*

###Answer
**Primary Metric:** Precision@K (evaluated at K=50)  
**Why this metric:** Content and editorial teams have limited operational capacity and evaluate candidates in fixed batches per sprint (represented by K). Precision@K measures what percentage of the top K recommended URLs in the queue are actual high-priority declining pages requiring attention.  
**What means "good":** A Precision@50 above 0.50 (50%). The rule-based baseline on the starter dataset achieves a Precision@50 of ~0.240 (24%). Reaching 50%+ at K=50 ensures the model doubles audit efficiency compared to static rules.

---



## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
import os
import sys
import subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df['target_is_declining'] = (df['trend_direction'] == 'down').astype(int)

sample_cols = [
    'content_id',
    'client_id',
    'content_age_days',
    'impressions_90d',
    'clicks_90d',
    'avg_position',
    'ctr',
    'target_is_declining'
]

print(f"Unit of Analysis: 1 Row = 1 Content Item (content_id)")
print(f"Dataframe Shape: {df.shape[0]:,} rows x {df.shape[1]} columns\n")

df[sample_cols].head(5)

Unit of Analysis: 1 Row = 1 Content Item (content_id)
Dataframe Shape: 30,000 rows x 45 columns



,content_id,client_id,content_age_days,impressions_90d,clicks_90d,avg_position,ctr,target_is_declining
0,content_304f48230142,client_f369cb89fc,187,3803,29,10.6,0.76,1
1,content_a1fb4e703a9e,client_4e07408562,445,15320,7,20.3,0.05,1
2,content_9aa793d4d895,client_7f2253d7e2,141,12581,11,36.5,0.09,1
3,content_331d6c4de07b,client_19581e27de,463,11751,58,6.2,0.49,0
4,content_d99b7a2d90ca,client_3fdba35f04,263,19140,24,44.0,0.13,1


**Unit of Analysis:**

One row = one web page (represented by a pseudonymized content_id) evaluated over a trailing 90-day window.

---



## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
fixed_rule = df[(df['impressions_90d'] >= 500) & (df['trend_direction'] == 'down')]

rule_count = len(fixed_rule)
total_count = len(df)

print(f"Total catalog evaluated: {total_count:,} pages")
print(f"Pages flagged by fixed rule: {rule_count:,} ({rule_count/total_count:.1%})")

Total catalog evaluated: 30,000 pages
Pages flagged by fixed rule: 9,961 (33.2%)


**Fixed rules** (such as if impressions >= 500 AND trend == down) produce flat binary flags that fail to prioritize workload. A fixed rule cannot sort thousands of flagged URLs by urgency or adjust to weekly editorial review capacity.  

**Machine learning** beats fixed rules because search performance decay involves non-linear interactions across multiple signals—such as ranking position drift, impression volume changes, and click-through rates. An ML model combines these messy, tangled signals into a single continuous priority score, allowing us to generate an ordered queue where high-exposure pages facing severe decay are surfaced first.

---



## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.